## RAG Day 5

### Let's go PRO! — Advanced RAG Techniques

Time to level up. We'll rebuild the pipeline with four upgrades:

1. **No LangChain!** Just native OpenAI + Chroma, for maximum flexibility and to see what's really going on.
2. **LLM-driven chunking** — instead of blindly cutting every N characters, we let an LLM split each document into *sensible* overlapping chunks.
3. **Document pre-processing** — for each chunk the LLM also writes a **headline** and a **summary**, which we store alongside the original text so retrieval has more to match against.
4. **Advanced retrieval** — **query rewriting** (turn a chatty question into a sharp search query) and **reranking** (have an LLM reorder the retrieved chunks by true relevance).

We keep the best chunk size / embedding model from the previous days.

In [1]:
from pathlib import Path

from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
from litellm import completion
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go

In [2]:
MODEL = "gpt-4.1-nano"
embedding_model = "text-embedding-3-large"
collection_name = "docs"
AVERAGE_CHUNK_SIZE = 500
RETRIEVAL_K = 10

# Resolve paths relative to lectures/week-five, so this runs from the repo root
# or from the notebook's own folder.
for candidate in [Path.cwd(), *Path.cwd().parents]:
    possible = candidate / "lectures" / "week-five"
    if (possible / "knowledge-base").exists():
        WEEK_DIR = possible
        break
else:
    WEEK_DIR = Path.cwd()

KNOWLEDGE_BASE_PATH = WEEK_DIR / "knowledge-base"
DB_NAME = str(WEEK_DIR / "preprocessed_db")  # a new, separate store for our pre-processed chunks

load_dotenv(override=True)
openai = OpenAI()

### Two little data classes

We're not using LangChain, so we'll make our own. `Result` mirrors LangChain's `Document` (page content + metadata). `Chunk` is what we ask the LLM to produce for each piece of a document.

In [3]:
# Inspired by LangChain's Document - let's have something similar

class Result(BaseModel):
    page_content: str
    metadata: dict

In [4]:
# A class to perfectly represent a chunk

class Chunk(BaseModel):
    headline: str = Field(description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query")
    summary: str = Field(description="A few sentences summarizing the content of this chunk to answer common questions")
    original_text: str = Field(description="The original text of this chunk from the provided document, exactly as is, not changed in any way")

    def as_result(self, document):
        metadata = {"source": document["source"], "type": document["type"]}
        return Result(page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text, metadata=metadata)


class Chunks(BaseModel):
    chunks: list[Chunk]

## Three steps

1. Fetch documents from the knowledge base (like LangChain's loader did)
2. Call an LLM to turn each document into `Chunks`
3. Store the chunks in Chroma

That's it!

### Step 1 — fetch the documents

In [5]:
def fetch_documents():
    """A homemade version of the LangChain DirectoryLoader"""
    documents = []
    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name
        for file in folder.rglob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append({"type": doc_type, "source": file.as_posix(), "text": f.read()})
    print(f"Loaded {len(documents)} documents")
    return documents

In [6]:
documents = fetch_documents()

Loaded 76 documents


### Step 2 — make the chunks with an LLM

We build a prompt that asks the model to split a document into roughly the right number of overlapping chunks, and to write a headline + summary + verbatim original text for each. We suggest a chunk count based on `AVERAGE_CHUNK_SIZE`, but let the model use judgement.

In [7]:
def make_prompt(document):
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    return f"""
You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: {document["type"]}
The document has been retrieved from: {document["source"]}

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into {how_many} chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

{document["text"]}

Respond with the chunks.
"""

In [8]:
print(make_prompt(documents[0]))


You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: products
The document has been retrieved from: /Users/marcolerma/Personal/GitHub-marclerm/applied-llm-engineering/lectures/week-five/knowledge-base/products/Rellm.md

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into 8 chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document wi

In [9]:
def make_messages(document):
    return [
        {"role": "user", "content": make_prompt(document)},
    ]

In [10]:
def process_document(document):
    messages = make_messages(document)
    response = completion(model=MODEL, messages=messages, response_format=Chunks)
    reply = response.choices[0].message.content
    doc_as_chunks = Chunks.model_validate_json(reply).chunks
    return [chunk.as_result(document) for chunk in doc_as_chunks]

In [11]:
process_document(documents[0])

[Result(page_content='Introduction to Rellm\n\nRellm is an AI-powered enterprise reinsurance platform developed by Insurellm, designed to enhance risk management, decision-making, and operational efficiency.\n\n# Rellm: AI-Powered Enterprise Reinsurance Solution\n\n## Summary\n\nRellm is an innovative enterprise reinsurance product developed by Insurellm, designed to transform the way reinsurance companies operate. Harnessing the power of artificial intelligence, Rellm offers an advanced platform that redefines risk management, enhances decision-making processes, and optimizes operational efficiencies within the reinsurance industry. With seamless integrations and robust analytics, Rellm enables insurers to proactively manage their portfolios and respond to market dynamics with agility.', metadata={'source': '/Users/marcolerma/Personal/GitHub-marclerm/applied-llm-engineering/lectures/week-five/knowledge-base/products/Rellm.md', 'type': 'products'}),
 Result(page_content="Key Features o

Now run it across every document. This is the slow part — one LLM call per document.

In [12]:
def create_chunks(documents):
    chunks = []
    for doc in tqdm(documents):
        chunks.extend(process_document(doc))
    return chunks

In [13]:
chunks = create_chunks(documents)

100%|██████████| 76/76 [07:59<00:00,  6.31s/it]


In [14]:
print(len(chunks))

381


### Step 3 — embed and store the chunks

We embed every chunk's full `page_content` (headline + summary + original text) and store the vectors in a fresh Chroma collection. Re-running wipes the old collection first so we don't accumulate duplicates.

In [15]:
def create_embeddings(chunks):
    chroma = PersistentClient(path=DB_NAME)
    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)

    texts = [chunk.page_content for chunk in chunks]
    emb = openai.embeddings.create(model=embedding_model, input=texts).data
    vectors = [e.embedding for e in emb]

    collection = chroma.get_or_create_collection(collection_name)
    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]

    collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
    print(f"Vectorstore created with {collection.count()} documents")

In [16]:
create_embeddings(chunks)

Vectorstore created with 381 documents


## Nothing more to do here... right?

Wait! Didja think I'd forget the visualization? 😄 Let's project the pre-processed vectors down with t-SNE and see how the four document types cluster.

In [17]:
chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_or_create_collection(collection_name)
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

In [18]:
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [19]:
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()

## And now — let's build an Advanced RAG!

Two techniques layered on top of basic retrieval:

1. **Reranking** — reorder the retrieved results with an LLM so the most relevant chunk comes first.
2. **Query rewriting** — turn the user's (possibly chatty) question into a sharp search query before we retrieve.

### Reranking

We ask an LLM to re-order the retrieved chunk ids by true relevance to the question.

In [20]:
class RankOrder(BaseModel):
    order: list[int] = Field(
        description="The order of relevance of chunks, from most relevant to least relevant, by chunk id number"
    )

In [21]:
def rerank(question, chunks):
    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked.
"""
    user_prompt = f"The user has asked the following question:\n\n{question}\n\nOrder all the chunks of text by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
    user_prompt += "Here are the chunks:\n\n"
    for index, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Reply only with the list of ranked chunk ids, nothing else."
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    response = completion(model=MODEL, messages=messages, response_format=RankOrder)
    reply = response.choices[0].message.content
    order = RankOrder.model_validate_json(reply).order
    print(order)
    return [chunks[i - 1] for i in order]

### Basic (unranked) retrieval

Native Chroma query: embed the question, fetch the top-k nearest chunks.

In [22]:
def fetch_context_unranked(question):
    query = openai.embeddings.create(model=embedding_model, input=[question]).data[0].embedding
    results = collection.query(query_embeddings=[query], n_results=RETRIEVAL_K)
    chunks = []
    for result in zip(results["documents"][0], results["metadatas"][0]):
        chunks.append(Result(page_content=result[0], metadata=result[1]))
    return chunks

Let's see reranking in action. First, retrieve unranked:

In [23]:
question = "Who won the IIOTY award?"
chunks = fetch_context_unranked(question)

In [24]:
for chunk in chunks:
    print(chunk.page_content[:15] + "...")

Career Progress...
Career Progress...
Annual Performa...
Avery Lancaster...
Priya Sharma's ...
Career Progress...
Annual Performa...
Career Progress...
Performance and...
Performance His...


In [25]:
reranked = rerank(question, chunks)

[1, 4, 2, 6, 8, 3, 7, 9, 10, 5]


In [26]:
for chunk in reranked:
    print(chunk.page_content[:15] + "...")

Career Progress...
Avery Lancaster...
Career Progress...
Career Progress...
Career Progress...
Annual Performa...
Annual Performa...
Performance and...
Performance His...
Priya Sharma's ...


A nicer demonstration: ask about Manchester University. Retrieve 20 chunks and find where the relevant one lands **before** reranking...

In [39]:
question = "Who went to Manchester University?"
RETRIEVAL_K = 20
chunks = fetch_context_unranked(question)
for index, c in enumerate(chunks):
    if "manchester" in c.page_content.lower():
        print(index)

...and where it lands **after** reranking (should jump much closer to the top):

In [40]:
reranked = rerank(question, chunks)

[8, 1, 2, 3, 4, 5, 6, 7, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]


In [42]:
for index, c in enumerate(reranked):
    if "manchester" in c.page_content.lower():
        print(index)

In [43]:
reranked[0].page_content

"Priya Sharma's Career Progression at Insurellm and Previous Roles (Part 1)\n\nDetails Priya Sharma's career at Insurellm from March 2018 onwards, highlighting her leadership in machine learning projects and mentorship, as well as her prior role at FinML Analytics.\n\n## Insurellm Career Progression\n- **March 2018 - Present:** Senior Data Scientist\n  - Leads machine learning initiatives for risk prediction models\n  - Built recommendation engine for Marketllm increasing conversion by 28%\n  - Mentors team of 3 junior data scientists\n  - Published 2 research papers on insurance ML applications\n\n- **June 2015 - February 2018:** Data Scientist at FinML Analytics\n  - Developed predictive models for financial services clients\n  - Specialized in customer churn prediction and fraud detection"

### Put retrieval together: fetch then rerank

In [31]:
def fetch_context(question):
    chunks = fetch_context_unranked(question)
    return rerank(question, chunks)

### The RAG prompt

We also include the **source** of each chunk in the context, so the model can ground its answer.

In [32]:
SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.
If you don't know the answer, say so.
For context, here are specific extracts from the Knowledge Base that might be directly relevant to the user's question:
{context}

With this context, please answer the user's question. Be accurate, relevant and complete.
"""

In [33]:
# In the context, include the source of the chunk

def make_rag_messages(question, history, chunks):
    context = "\n\n".join(f"Extract from {chunk.metadata['source']}:\n{chunk.page_content}" for chunk in chunks)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    return [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": question}]

### Query rewriting

Turn the user's question into a short, specific search query that's more likely to surface the right chunks.

In [34]:
def rewrite_query(question, history=[]):
    """Rewrite the user's question to be a more specific question that is more likely to surface relevant content in the Knowledge Base."""
    message = f"""
You are in a conversation with a user, answering questions about the company Insurellm.
You are about to look up information in a Knowledge Base to answer the user's question.

This is the history of your conversation so far with the user:
{history}

And this is the user's current question:
{question}

Respond only with a single, refined question that you will use to search the Knowledge Base.
It should be a VERY short specific question most likely to surface content. Focus on the question details.
Don't mention the company name unless it's a general question about the company.
IMPORTANT: Respond ONLY with the knowledgebase query, nothing else.
"""
    response = completion(model=MODEL, messages=[{"role": "system", "content": message}])
    return response.choices[0].message.content

In [35]:
rewrite_query("Who won the IIOTY award?", [])

'Who received the IIOTY award?'

### The full advanced pipeline

`rewrite_query` → `fetch_context` (retrieve + rerank) → answer. Returns the answer and the chunks that grounded it.

In [36]:
def answer_question(question: str, history: list[dict] = []) -> tuple[str, list]:
    """Answer a question using RAG and return the answer and the retrieved context."""
    query = rewrite_query(question, history)
    print(query)
    chunks = fetch_context(query)
    messages = make_rag_messages(question, history, chunks)
    response = completion(model=MODEL, messages=messages)
    return response.choices[0].message.content, chunks

In [37]:
answer_question("Who won the IIOTY award?", [])

Who was the recipient of the IIOTY award?
[1, 4, 9, 5, 14, 11, 15, 6, 7, 10, 16, 12, 13, 2, 3, 8, 17, 18, 19, 20]


('Maxine Thompson was recognized as the Insurellm Innovator of the Year (IIOTY) in 2023.',
 [Result(page_content="Career Progression at Insurellm (Part 2)\n\nThis segment continues Maxine Thompson's career growth at Insurellm from 2021 onwards, highlighting her promotion, projects, and recognition.\n\n- **January 2021 - Present**: **Senior Data Engineer**  \t* Maxine was promoted to Senior Data Engineer after successfully leading a pivotal project that improved data retrieval times by 30%. She now mentors junior engineers and is involved in strategic data initiatives, solidifying her position as a valued asset at Insurellm. She was recognized as Insurellm Innovator of the year in 2023, receiving the prestigious IIOTY 2023 award.", metadata={'type': 'employees', 'source': '/Users/marcolerma/Personal/GitHub-marclerm/applied-llm-engineering/lectures/week-five/knowledge-base/employees/Maxine Thompson.md'}),
  Result(page_content="Avery Lancaster's Annual Performance History Overview\n\nThi

In [38]:
answer_question("Who went to Manchester University?", [])

Which individuals attended Manchester University?
[7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 1, 2, 3, 4, 5, 6, 20]


('Based on the provided HR records and employee information, there is no record of anyone having attended Manchester University.',
 [Result(page_content='HR Record of Samuel Trenton\n\nThis section introduces Samuel Trenton, detailing his personal and professional baseline information within Insurellm.\n\n# HR Record\n\n# Samuel Trenton', metadata={'type': 'employees', 'source': '/Users/marcolerma/Personal/GitHub-marclerm/applied-llm-engineering/lectures/week-five/knowledge-base/employees/Samuel Trenton.md'}),
  Result(page_content="Educational Background, Certifications, and Recognitions\n\nAmanda holds a master's degree from Cornell University and a bachelor's from UCLA. She is certified as a SHRM-SCP and a Certified Diversity Professional, and received the HR Excellence Award in 2023.\n\n## Other HR Notes\n- **Education:** MS in Human Resources Management from Cornell University, BA in Psychology from UCLA\n- **Certifications:** SHRM-SCP, Certified Diversity Professional\n- **Recogn

## That's a wrap on Advanced RAG! 🎉

We went fully native — LLM-driven semantic chunking, document pre-processing (headline + summary), query rewriting, and reranking. These four techniques are exactly the kind of upgrades that turn a demo RAG into a reliable production one. Use the Day 4 evaluation dashboard to prove each one actually helps!